# ForenSight: Colab quick start

Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom.

1. Setup (GPU check, Drive, install)
2. **Audio, no approval needed:** download ASVspoof 2019 LA, evaluate the pretrained AASIST, train the MelCNN baseline
3. **Video, after FaceForensics++ access is approved:** preprocess faces, train EfficientNet-B4

Results are written to `MyDrive/forensight/runs/`. Log each finished run in `docs/experiments.md`.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU (CPU runtime): fine for downloads, not for training/scoring"
import os, shutil
from google.colab import drive
if not os.path.ismount("/content/drive"):
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        # something was written here before Drive was connected; keep it aside instead of failing
        shutil.move("/content/drive", "/content/drive_leftover")
        print("Moved files written before Drive was connected to /content/drive_leftover")
    drive.mount("/content/drive")
else:
    print("Drive already connected")

In [ ]:
# The repo is private. Create a GitHub fine-grained token (Settings > Developer settings >
# Fine-grained tokens, repo ForenSight-FYP, Contents: read) and save it in Colab's
# Secrets panel (key icon on the left) as GITHUB_TOKEN with notebook access on.
from google.colab import userdata
TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{TOKEN}@github.com/ForenSight-FYP/ForenSight-FYP.git"
!git clone -q $REPO_URL /content/forensight || (cd /content/forensight && git pull -q)
%cd /content/forensight
!pip install -q -e ".[dl,dev]"
!python scripts/fix_opencv.py
!pytest -q

## 2. Audio module

### 2a. ASVspoof 2019 LA **evaluation** set (Hugging Face copy, ~4.4 GB, about 1 minute)
Edinburgh DataShare returns **403 Forbidden** to Colab, so the eval set comes from
`SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA` on Hugging Face: the official eval audio and the
official CM eval protocol, unmodified, redistributed under the same ODC-By licence.
The audio is stored in Parquet shards; this cell writes it back out as FLAC files.

In [ ]:
import os, glob, pyarrow.parquet as pq
from huggingface_hub import snapshot_download

OUT = "/content/asvspoof/LA19_eval"
snapshot_download("SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA", repo_type="dataset", local_dir=OUT)
os.makedirs(f"{OUT}/flac", exist_ok=True)
for pf in sorted(glob.glob(f"{OUT}/data/test-*.parquet")):
    table = pq.read_table(pf, columns=["audio"])
    for row in table.column("audio").to_pylist():
        name = os.path.basename(row["path"])
        name = name if name.endswith(".flac") else name + ".flac"
        with open(f"{OUT}/flac/{name}", "wb") as f:
            f.write(row["bytes"])
PROTO = glob.glob(f"{OUT}/**/ASVspoof2019.LA.cm.eval.trl.txt", recursive=True)[0]
n = len(os.listdir(f"{OUT}/flac"))
print("clips:", n, "| labels:", PROTO)
assert n == 71237, "expected 71,237 eval clips"
LA19 = f"eval_sets={{la19_eval: {{protocol: {PROTO}, audio: {OUT}/flac}}}}"
%cd /content/forensight

### 2b. Pretrained AASIST on ASVspoof 2019 LA eval (pipeline sanity check)
The AASIST paper reports EER about 0.83%. **Our first run (2026-11-17): EER 0.82%, AUC 0.9993**
on all 71,237 clips, so data, labels, model wrapper and metrics are correct. Takes 10-30 min on a T4.

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
!bash scripts/setup_aasist.sh
cmd = f'python -m forensight.evaluation.eval_audio --config configs/audio_aasist.yaml "{LA19}"'
!{cmd}

### 2c. Train the MelCNN baseline: needs the **train/dev** sets from the full `LA.zip`
The Hugging Face copy has the eval set only. The full `LA.zip` (7.6 GB) comes from the organisers' own
Zenodo record 6906306 (same DOI as the Edinburgh release, 10.7488/ds/2555, ODC-By licence), which works
from Colab. It is checked against the published MD5 and cached in Drive if there is room.

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
import os, shutil
URL = "https://zenodo.org/records/6906306/files/LA.zip?download=1"
MD5 = "30c98f11d8b2bc21f2c257bfd78bb5c5"
CACHE = "/content/drive/MyDrive/forensight/datasets/asvspoof2019_LA.zip"
LOCAL = "/content/LA.zip"

def md5_ok(path):
    if not os.path.exists(path):
        return False
    out = !md5sum "{path}"
    return out[0].split()[0] == MD5

if md5_ok(CACHE):
    print("Using verified LA.zip from Drive"); src = CACHE
else:
    if os.path.exists(CACHE):   # broken or partial earlier copy
        os.remove(CACHE)
    !wget -c --progress=dot:giga -O {LOCAL} "{URL}" 2>&1 | tail -5
    assert md5_ok(LOCAL), "LA.zip download is incomplete or corrupted: delete /content/LA.zip and re-run"
    src = LOCAL
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    free = shutil.disk_usage("/content/drive/MyDrive").free
    if free > 9 * 1024**3:
        !cp {LOCAL} {CACHE}
        print("Cached in Drive for next time")
    else:
        print(f"Only {free/1024**3:.1f} GB free in Drive: not caching (it re-downloads next session)")
!mkdir -p /content/asvspoof && unzip -q -n {src} -d /content/asvspoof
!ls /content/asvspoof/LA

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
# smoke run first: 1 epoch, no extra test sets
!python -m forensight.training.train_audio --config configs/audio_melcnn.yaml train.epochs=1 eval_sets=null output_dir=/content/drive/MyDrive/forensight/runs/audio_melcnn_smoke

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
# full run (about 20 epochs), tested on the 2019 LA eval set from 2a
cmd = f'python -m forensight.training.train_audio --config configs/audio_melcnn.yaml "{LA19}"'
!{cmd}

### 2d. ASVspoof 2021 DF: cross-dataset test (generalisation, objective O3)
Official DF eval audio: Zenodo record 4835108, four parts of ~8.6 GB (34.5 GB, ~611,829 clips), plus
the official keys from asvspoof.org. Each part is checked against its MD5, cached in Drive, extracted,
and marked done, so re-running after a disconnect only does what is left.

**Recommended order:** run with `PARTS = ["00"]` first and score it (a quick subset result), then set
all four parts and run again for the full set. Scoring is **chunked**: every 20,000 clips are saved to
Drive, and a re-run continues from the last saved chunk.

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
import os, subprocess
PARTS = ["00"]          # first run; later: ["00", "01", "02", "03"] for the full set
MD5 = {"00": "4f2cbae07cf3ede2a1dde3b8d2ee55ea", "01": "1578c89ab433c2b60b1dce93bdf8fbec",
       "02": "5497a35f0126e94a1d7a7d26db57b4f7", "03": "42b7512ba2943e98a32a53c9608cf83c"}
ROOT = "/content/asvspoof/DF21"
CACHE_DIR = "/content/drive/MyDrive/forensight/datasets/asvspoof2021_DF"
os.makedirs(ROOT, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

def md5_ok(path, part):
    if not os.path.exists(path):
        return False
    out = subprocess.run(["md5sum", path], capture_output=True, text=True).stdout
    return out.split()[0] == MD5[part] if out else False

%cd {ROOT}
!wget -q -c https://www.asvspoof.org/asvspoof2021/DF-keys-full.tar.gz && tar -xzf DF-keys-full.tar.gz

for part in PARTS:
    f = f"ASVspoof2021_DF_eval_part{part}.tar.gz"
    done = f"{ROOT}/.done_part{part}"      # marker: this part is fully extracted
    cached = f"{CACHE_DIR}/{f}"
    if os.path.exists(done):
        print(f"part{part}: already extracted, skipping")
        continue
    if os.path.exists(cached + ".part"):   # leftover from an interrupted Drive copy
        os.remove(cached + ".part")
    from_drive = False
    if os.path.exists(cached):
        print(f"part{part}: copying from Drive")
        !cp "{cached}" {f}
        from_drive = md5_ok(f, part)
        if not from_drive:
            print(f"part{part}: Drive copy is corrupted, removing it and downloading instead")
            os.remove(cached)
            os.remove(f)
    if not from_drive:
        print(f"part{part}: downloading from Zenodo")
        !wget -q -c --show-progress -O {f} "https://zenodo.org/records/4835108/files/{f}?download=1"
        if not md5_ok(f, part):
            raise RuntimeError(f"{f} is corrupted: delete {ROOT}/{f} and re-run")
        print(f"part{part}: saving to Drive (several minutes, no progress shown)")
        !cp {f} "{cached}.part" && mv "{cached}.part" "{cached}"
    !tar -xzf {f} && rm {f} && touch {done}
    if not os.path.exists(done):
        raise RuntimeError(f"Extracting {f} failed (disk full?): check `!df -h /content` and re-run")
    print(f"part{part}: done")

%cd /content/forensight
DF_KEYS = !find /content/asvspoof/DF21 -name trial_metadata.txt -path "*/CM/*"
DF_AUDIO = !find /content/asvspoof/DF21 -name "DF_E_*.flac" -print -quit
assert DF_KEYS, "trial_metadata.txt not found: the keys download or extraction failed"
assert DF_AUDIO, "no DF_E_*.flac files found: the audio extraction failed"
DF_KEYS, DF_AUDIO = DF_KEYS[0], os.path.dirname(DF_AUDIO[0])
n = len(os.listdir(DF_AUDIO))
print(DF_KEYS, DF_AUDIO, n, "clips")
if len(PARTS) == 4 and n < 611829:
    print(f"Warning: expected about 611,829 clips for the full set, got {n}")

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
# Pretrained AASIST on the DF clips present (subset = official "eval" trials).
# Chunks of 20,000 are saved to Drive; if Colab disconnects, re-run Setup, 2d's first cell, then this cell.
# Expect a much higher EER than on 2019 LA: that gap is the generalisation problem in the problem statement.
!bash scripts/setup_aasist.sh
OVR = f"eval_sets={{df21_eval: {{protocol: {DF_KEYS}, audio: {DF_AUDIO}, subset: eval, only_existing: true}}}}"
cmd = f'python -m forensight.evaluation.eval_audio --config configs/audio_aasist.yaml chunk_size=20000 "{OVR}"'
!{cmd}

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
# your MelCNN on the same clips (after 2c has trained it)
RUN = "/content/drive/MyDrive/forensight/runs/audio_melcnn"
cmd = (f'python -m forensight.evaluation.eval_audio --config configs/audio_aasist.yaml chunk_size=20000 '
       f'model.name=melcnn model.checkpoint={RUN}/best.pt output_dir={RUN} "{OVR}"')
!{cmd}

## 3. Video module (FaceForensics++ c23)

**3a. Face crops (CPU is fine, about 1-3 hours).** Copies the 5,000 FF++ videos from Drive to the fast
local disk, crops 16 faces per video, and saves one archive per split to
`MyDrive/forensight/datasets/ffpp_c23_faces/` (`train.tar`, `val.tar`, `test.tar`). Resumable: re-run
after a disconnect and finished videos and finished splits are skipped.

**3b. Training (GPU).** Restores the crops from those archives (a few minutes) and trains.
Anyone with the shared `forensight` folder can run 3b, so one person can crop and another train.

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
FF = "/content/drive/MyDrive/forensight/datasets/FaceForensics++"
LOCAL_FF = "/content/ffpp_local"
!python -c "import mediapipe; print('mediapipe', mediapipe.__version__)" || pip install -q mediapipe
!python scripts/fix_opencv.py
# local copy of the videos: much faster to read than Drive (about 10 GB, several minutes)
!mkdir -p {LOCAL_FF} && cp -rn "{FF}/original_sequences" "{FF}/manipulated_sequences" {LOCAL_FF}/
!find {LOCAL_FF} -name "*.mp4" | wc -l
!python -m forensight.data.video_preprocess --config configs/video_effnet.yaml data.ffpp_root={LOCAL_FF}
!ls -lh /content/drive/MyDrive/forensight/datasets/ffpp_c23_faces/

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
# 3b. restore crops from Drive (fast), then a smoke run on a small backbone
!python -m forensight.data.video_preprocess --config configs/video_effnet.yaml
!python -m forensight.training.train_video --config configs/video_effnet.yaml train.epochs=1 model.backbone=efficientnet_b0 data.image_size=224 output_dir=/content/drive/MyDrive/forensight/runs/video_smoke

In [ ]:
import os
assert os.path.ismount("/content/drive"), "Google Drive is not connected: run the first Setup cell first"
# full run: EfficientNet-B4, about 8 epochs (several hours on a T4).
# Resumable: after a disconnect, run Setup and this cell again; it continues from the last finished epoch.
!python -m forensight.data.video_preprocess --config configs/video_effnet.yaml
!python -m forensight.training.train_video --config configs/video_effnet.yaml